# Fine-tune MiniLM on BEIR SciFact

Fine-tunes `sentence-transformers/all-MiniLM-L6-v2` (the app's embedding model) on SciFact's
**train** split with `MultipleNegativesRankingLoss` and BM25 hard negatives, picks the epoch count
on a held-out dev split of the train queries, and saves the best model as a zip.

The **test** split is never used here; it is evaluated locally with
`python benchmarks/benchmark_scifact.py --model <unzipped folder> --label fine-tuned`.

**Run on Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all
(about 5–10 minutes). The zip downloads at the end.

In [ ]:
# Colab only. Same sentence-transformers / rank-bm25 versions as the project.
%pip install -q sentence-transformers==6.1.0 rank-bm25==0.2.2 datasets accelerate

## Settings and seeds

In [ ]:
import json, math, os, random, re, shutil, sys, urllib.request, zipfile

import numpy as np
import torch
from datasets import Dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import (
    SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.training_args import BatchSamplers
from transformers import set_seed

SEED = 42
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MAX_EPOCHS = 3            # the best of epochs 1..MAX_EPOCHS is chosen on the dev split
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WARMUP_FRACTION = 0.1
DEV_FRACTION = 0.1        # share of train queries held out for choosing the epoch
NUM_HARD_NEGATIVES = 2    # per (query, positive) pair, mined with BM25
HARD_NEGATIVE_DEPTH = 20  # look for negatives in BM25's top 20
OUTPUT_DIR = "minilm-scifact"
DATASET_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"

# SMOKE_TEST=1 trains on a tiny subset, to check the notebook runs end to end on a CPU.
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| smoke test:", SMOKE_TEST)

## Load SciFact

In [ ]:
if not os.path.exists("scifact/corpus.jsonl"):
    urllib.request.urlretrieve(DATASET_URL, "scifact.zip")
    zipfile.ZipFile("scifact.zip").extractall(".")

# Title + abstract, joined exactly as in benchmarks/benchmark_scifact.py.
corpus = {}
with open("scifact/corpus.jsonl", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        corpus[d["_id"]] = f"{d['title']}\n{d['text']}".strip()

queries = {}
with open("scifact/queries.jsonl", encoding="utf-8") as f:
    for line in f:
        q = json.loads(line)
        queries[q["_id"]] = q["text"]

def load_qrels(split):
    qrels = {}
    with open(f"scifact/qrels/{split}.tsv", encoding="utf-8") as f:
        next(f)
        for line in f:
            qid, doc_id, score = line.rstrip("\n").split("\t")
            if int(score) > 0:
                qrels.setdefault(qid, set()).add(doc_id)
    return qrels

train_qrels = load_qrels("train")
test_qrels = load_qrels("test")
print(f"{len(corpus)} documents, {len(train_qrels)} train queries, {len(test_qrels)} test queries")

## Leakage check: no test query in the training data

The official split shares no query ids, but two train claims are word-for-word copies of test
claims with the same relevant document (train 871 = test 870, train 1291 = test 1292). Training on
them would leak test answers, so any train query whose text matches a test query is dropped first.

In [ ]:
def normalise(text):
    return " ".join(text.lower().split())

test_ids = set(test_qrels)
test_texts = {normalise(queries[q]) for q in test_ids}
duplicates = sorted((q for q in train_qrels if normalise(queries[q]) in test_texts), key=int)
for q in duplicates:
    print(f"dropping train query {q} (same text as a test query): {queries[q]}")
    del train_qrels[q]

assert not test_ids & set(train_qrels), "a test query id is in the train qrels"
assert not any(normalise(queries[q]) in test_texts for q in train_qrels), "a test query text is in train"
print(f"OK: none of the {len(test_ids)} test queries (by id or by text) remain in the "
      f"{len(train_qrels)} train queries.")

## Split train queries into train and dev

In [ ]:
train_ids = sorted(train_qrels, key=int)
random.Random(SEED).shuffle(train_ids)
n_dev = round(len(train_ids) * DEV_FRACTION)
dev_ids, fit_ids = train_ids[:n_dev], train_ids[n_dev:]
if SMOKE_TEST:
    # A batch of 32 rows is ~128 sequences of up to 256 tokens, which runs out of
    # memory on a laptop CPU; the T4 handles it easily.
    fit_ids, dev_ids, MAX_EPOCHS, BATCH_SIZE = fit_ids[:48], dev_ids[:16], 2, 8
print(f"{len(fit_ids)} queries for training, {len(dev_ids)} for dev")

## Mine BM25 hard negatives

For each training query, the top-ranked BM25 documents that are **not** labelled relevant.
The tokenizer is the project's (`src/vector_store.py`): lowercase, `\b[\w-]+\b`.
SciFact labels are sparse, so a "negative" can occasionally be an unlabelled relevant document;
taking only 2 per pair keeps that noise small.

In [ ]:
TOKEN_PATTERN = re.compile(r"\b[\w-]+\b")

def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())

doc_ids = list(corpus)
bm25 = BM25Okapi([tokenize(corpus[d]) for d in doc_ids])

rows = {"anchor": [], "positive": []}
rows.update({f"negative_{i + 1}": [] for i in range(NUM_HARD_NEGATIVES)})
for qid in fit_ids:
    scores = bm25.get_scores(tokenize(queries[qid]))
    top = np.argsort(-scores)[:HARD_NEGATIVE_DEPTH]
    negatives = [doc_ids[i] for i in top if doc_ids[i] not in train_qrels[qid]][:NUM_HARD_NEGATIVES]
    for pos in sorted(train_qrels[qid]):
        rows["anchor"].append(queries[qid])
        rows["positive"].append(corpus[pos])
        for i, neg in enumerate(negatives):
            rows[f"negative_{i + 1}"].append(corpus[neg])

train_dataset = Dataset.from_dict(rows)
print(train_dataset)
print("example anchor:", train_dataset[0]["anchor"])

## Dev evaluator

Retrieval over the full corpus for the dev queries. The model reads the first 256 tokens of each
document here; the local benchmark uses the app's chunks, so its numbers will differ a little.

In [ ]:
dev_corpus = corpus
if SMOKE_TEST:  # a small corpus that still contains every dev positive
    keep = {d for q in dev_ids for d in train_qrels[q]} | set(doc_ids[:300])
    dev_corpus = {d: corpus[d] for d in keep}

dev_evaluator = InformationRetrievalEvaluator(
    queries={q: queries[q] for q in dev_ids},
    corpus=dev_corpus,
    relevant_docs={q: train_qrels[q] for q in dev_ids},
    name="scifact-dev",
    batch_size=64,
    show_progress_bar=False,
)
NDCG_KEY = "scifact-dev_cosine_ndcg@10"
MRR_KEY = "scifact-dev_cosine_mrr@10"

model = SentenceTransformer(BASE_MODEL, device=device)
base_scores = dev_evaluator(model)
print(f"base model on dev: nDCG@10 {base_scores[NDCG_KEY]:.4f}, MRR@10 {base_scores[MRR_KEY]:.4f}")

## Fine-tune with MultipleNegativesRankingLoss

In [ ]:
steps_per_epoch = math.ceil(len(train_dataset) / BATCH_SIZE)
args = SentenceTransformerTrainingArguments(
    output_dir=f"{OUTPUT_DIR}-checkpoints",
    num_train_epochs=MAX_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=int(WARMUP_FRACTION * steps_per_epoch * MAX_EPOCHS),
    fp16=device == "cuda",
    # In-batch negatives: two copies of one positive in a batch would be false negatives.
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model=f"eval_{NDCG_KEY}",
    greater_is_better=True,
    logging_steps=10,
    seed=SEED,
    report_to="none",
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=losses.MultipleNegativesRankingLoss(model),
    evaluator=dev_evaluator,
)
trainer.train()

## Base vs fine-tuned on dev

In [ ]:
epochs = [
    {"epoch": round(log["epoch"]), "ndcg@10": log[f"eval_{NDCG_KEY}"], "mrr@10": log[f"eval_{MRR_KEY}"]}
    for log in trainer.state.log_history if f"eval_{NDCG_KEY}" in log
]
best = max(epochs, key=lambda e: e["ndcg@10"])
final_scores = dev_evaluator(trainer.model)  # the best checkpoint, reloaded at the end of training

print("| Model | dev nDCG@10 | dev MRR@10 |")
print("|---|---|---|")
print(f"| Base | {base_scores[NDCG_KEY]:.4f} | {base_scores[MRR_KEY]:.4f} |")
for e in epochs:
    print(f"| Epoch {e['epoch']} | {e['ndcg@10']:.4f} | {e['mrr@10']:.4f} |")
print(f"\nBest epoch: {best['epoch']}. Reloaded best model on dev: nDCG@10 {final_scores[NDCG_KEY]:.4f}")
if final_scores[NDCG_KEY] <= base_scores[NDCG_KEY]:
    print("WARNING: no epoch beat the base model on dev; fine-tuning did not help with these settings.")

## Save and download the best model

In [ ]:
trainer.model.save(OUTPUT_DIR)
summary = {
    "base_model": BASE_MODEL, "seed": SEED, "train_queries": len(fit_ids), "dev_queries": len(dev_ids),
    "train_pairs": len(train_dataset), "hard_negatives_per_pair": NUM_HARD_NEGATIVES,
    "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "warmup_fraction": WARMUP_FRACTION,
    "max_epochs": MAX_EPOCHS, "best_epoch": best["epoch"], "device": device, "smoke_test": SMOKE_TEST,
    "dev_base": {"ndcg@10": base_scores[NDCG_KEY], "mrr@10": base_scores[MRR_KEY]},
    "dev_per_epoch": epochs,
    "dev_finetuned": {"ndcg@10": final_scores[NDCG_KEY], "mrr@10": final_scores[MRR_KEY]},
}
with open(os.path.join(OUTPUT_DIR, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
shutil.rmtree(f"{OUTPUT_DIR}-checkpoints", ignore_errors=True)
archive = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("saved", archive)

if IN_COLAB:
    from google.colab import files
    files.download(archive)